### # ## ### In Databricks we implement Auto Loader inside notebooks using Structured Streaming. The notebook monitors a storage path, processes new files, and writes them to a Delta table while tracking progress using checkpoints.

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS catlod.auto_loader")

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.auto_loader.demo_table")
spark.sql("DROP TABLE IF EXISTS workspace.auto_loader.demo_table2")

In [0]:
df_stream=spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/auto_loader/auto_loader_files/schema/") \
  .load("/Volumes/workspace/auto_loader/auto_loader_files/source_files_autoload/")

In [0]:
df_stream.writeStream \
  .option("checkpointLocation", "/Volumes/workspace/auto_loader/auto_loader_files/checkpoint/") \
  .trigger(availableNow=True) \
  .format("json") \
  .start("/Volumes/workspace/auto_loader/auto_loader_files/output/") \
  .awaitTermination()

In [0]:
df_stream.writeStream \
  .option("checkpointLocation", "/Volumes/workspace/auto_loader/auto_loader_files/checkpoint5/") \
  .trigger(availableNow=True) \
  .toTable("workspace.auto_loader.demo_table2") \
  .awaitTermination()
  # here the loaction of check point is very important if we specify checkpoint it will fetch data at that 
  # particular check pointin order to get the data from the latest checkpoint use checkpoint 2,3,4....

In [0]:
%sql
select* from workspace.auto_loader.demo_table2

In [0]:
%sql
select * from cloud_files_state("/Volumes/workspace/auto_loader/auto_loader_files/checkpoint5/") 
-- to see the check point 

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("id", IntegerType(), nullable=False),
    StructField("customer_name", StringType(), nullable=True),
    StructField("product", StringType(), nullable=True),
    StructField("quantity", IntegerType(), nullable=True),
    StructField("price", IntegerType(), nullable=True),
    StructField("remarks", StringType(), nullable=True)
])

spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "json") \
  .schema(schema) \
  .option("cloudFiles.schemaLocation", "/Volumes/workspace/auto_loader/auto_loader_files/schema") \
  .load("/Volumes/workspace/auto_loader/auto_loader_files/source_files_autoload/") \
  .writeStream \
  .option("checkpointLocation", "/Volumes/workspace/auto_loader/auto_loader_files/checkpoint5") \
  .trigger(availableNow=True) \
  .toTable("workspace.auto_loader.demo_table2") \
  .awaitTermination()

In [0]:
spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "json") \
  .option("cloudFiles.inferSchema", "true") \
  .option("cloudFiles.schemaLocation", "/Volumes/workspace/auto_loader/auto_loader_files/schema") \
  .load("/Volumes/workspace/auto_loader/auto_loader_files/source_files_autoload/") \
  .writeStream \
  .option("checkpointLocation", "/Volumes/workspace/auto_loader/auto_loader_files/checkpoint4") \
  .trigger(availableNow=True) \
  .toTable("workspace.auto_loader.demo_table2") \
  .awaitTermination()

# schema evaluation rescue

In [0]:
spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "json") \
  .option("cloudFiles.inferColumnTypes", "true") \
  .option("cloudFiles.schemaEvolutionMode", "rescue") \
  .option("cloudFiles.schemaLocation", "/Volumes/workspace/auto_loader/auto_loader_files/schema/") \
  .load("/Volumes/workspace/auto_loader/auto_loader_files/source_files_autoload/")\
    .writeStream \
  .option("checkpointLocation", "/Volumes/workspace/auto_loader/auto_loader_files/checkpoint6") \
  .trigger(availableNow=True) \
  .table("workspace.auto_loader.demo_table5")\
  .awaitTermination()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("id", IntegerType(), nullable=False),
    StructField("customer_name", StringType(), nullable=True),
    StructField("product", StringType(), nullable=True),
    StructField("quantity", IntegerType(), nullable=True),
    StructField("price", IntegerType(), nullable=True),
    StructField("remarks", StringType(), nullable=True)
])

spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "json") \
  .schema(schema) \
  .option("cloudFiles.schemaEvolutionMode", "rescue") \
  .option("cloudFiles.schemaLocation", "/Volumes/workspace/auto_loader/auto_loader_files/schema1/") \
  .load("/Volumes/workspace/auto_loader/auto_loader_files/source_files_autoload/") \
  .writeStream \
  .option("checkpointLocation", "/Volumes/workspace/auto_loader/auto_loader_files/checkpoint1/") \
  .trigger(availableNow=True) \
  .toTable("workspace.auto_loader.demo_table2") \
  .awaitTermination()

### If you want Auto Loader to forget past data → change checkpoint
### If you want Auto Loader to forget past schema → change schemaLocation

### > file detection modes :
> 1) directory listing -- scan every file again and process only unprocessed files
> 
> 2) file notification mode -- the incoming file send a notification to autoloader to process 
> that file only no unnesessary scanning of all files 
> 
> 3. managed file events -- here data bricks configure the cloud events not us 

### > trigger types :
> 
> 1)available now -- process all available files and stops
> 
> 2)processing time -- runs continiously untill the specified time interval
> 
> 3)once -- process all data at once and then exists 

### > schema evaluation modes:

> 1)addNewColumns (Default — Most Used) -- slowly changing schema 
> .option("cloudFiles.schemaEvolutionMode","addNewColumns")
> 
> 2)rescue (Safe Mode)-- Unknown columns are NOT added to table.Instead stored inside special column:_rescued_data
> this is used when you dont want to fail pipeline but need that error data 
> .option("cloudFiles.schemaEvolutionMode","rescue")
> 
> 3)failOnNewColumns (Strict Mode)--If new column appears → pipeline FAILS immediately. strict evaluation 
> .option("cloudFiles.schemaEvolutionMode","failOnNewColumns")
> 
> 4)ignore -- Ignores new columns silently,No failure, no capture.
> .option("cloudFiles.schemaEvolutionMode","none")
> 
> 

In [0]:
spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "json") \
  .option("cloudFiles.inferSchema", "true") \
  .option("cloudFiles.schemaEvolutionMode", "rescue") \
  .option("cloudFiles.schemaLocation", "/Volumes/workspace/auto_loader/auto_loader_files/schema/") \
  .load("/Volumes/workspace/auto_loader/auto_loader_files/source_files_autoload/") \
  .writeStream \
  .option("checkpointLocation", "/Volumes/workspace/auto_loader/auto_loader_files/checkpoint2/") \
  .option("mergeSchema", "true") \
  .trigger(availableNow=True) \
  .toTable("workspace.auto_loader.demo_table2") \
  .awaitTermination()

In [0]:
%sql
select * from workspace.auto_loader.demo_table2

In [0]:
df_demo_table6 = spark.table("workspace.auto_loader.demo_table2")
display(df_demo_table6)

In [0]:
from pyspark.sql.functions import col, when

df_demo_table6 = df_demo_table6.withColumn(
    "highvalue_customer",
    when(col("quantity") * col("price") > 9000, True).otherwise(False)
)
display(df_demo_table6)

In [0]:
from pyspark.sql.functions import collect_list

df_highvalue_grouped = df_demo_table6.groupBy("highvalue_customer") \
    .agg(collect_list("customer_name").alias("customer_names"))

display(df_highvalue_grouped.select("highvalue_customer", "customer_names"))

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

data = [
    ("100", "3", "101", "2", "2026-04-01 09:15:21", None),
    ("101", "4", "102", "1", "2026-04-01 09:22:10", None),
    ("102", "5", "103", "4", "2026-04-01 09:40:55", None),
    ("103", "6", "104", "2", "2026-04-01 10:05:33", None),
    ("104", "7", "105", "3", "2026-04-01 10:18:02", None),
    ("105", "8", "106", "1", "2026-04-01 10:32:47", None),
    ("106", "9", "107", "5", "2026-04-01 10:55:19", None),
    ("107", "10", "108", "2", "2026-04-01 11:08:41", None),
    ("108", "3", "109", "3", "2026-04-01 11:26:12", None)
]

schema = StructType([
    StructField("purchase_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("product_id", StringType(), False),
    StructField("quantity", StringType(), False),
    StructField("purchase_time", StringType(), False),
    StructField("_rescued_data", StringType(), True)
])

df = spark.createDataFrame(data, schema)
df.write.format("delta").mode("overwrite").saveAsTable("workspace.delta_lake_scd.source_streaming_table")

In [0]:
%sql
select * from workspace.delta_lake_scd.source_streaming_table